# Brain Tumor MRI Leakage Benchmark — run everything

This notebook reproduces the full study: dataset audit, leakage detection, grouped and image-level cross-validation, baselines and ablations, final test evaluation, shortcut probes, Grad-CAM and CPU latency.

**Before running on Kaggle** (right-hand panel):

1. **Add Input → Datasets:** `masoudnickparvar/brain-tumor-mri-dataset` (version 2, 1,400/400 images per class).
2. **Session options:** Accelerator **GPU** (T4 or P100); **Internet On** (needed for the ImageNet weights, for cloning this repository and for the figshare patient-ID files).
3. Click **Save Version → Save & Run All (Commit)**.

The full run takes about 9–11 GPU hours. Steps run in priority order. Each finished run is saved, so if you attach the output of a previous version as an input, the notebook resumes instead of starting again.

**Output:** `results.zip` in the *Output* tab, containing all per-run results, summary values and figures.

In [ ]:
# ---- settings ------------------------------------------------------------
REPO_URL = "https://github.com/Asadullah-Irshad/Brain-Tumor-MRI-Leakage-Benchmark.git"
TIME_BUDGET_HOURS = 11.0          # Kaggle stops a commit at 12 h
import os, sys, time, glob, json, shutil, subprocess, zipfile
T0 = time.time()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "Brain-Tumor-MRI-Leakage-Benchmark")
OUT = os.path.join(WORK, "results")
SMOKE = os.environ.get("BENCH_SMOKE") == "1"    # fast end-to-end check only
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run(f"pip install -q onnxscript onnxruntime 2>/dev/null", shell=True)
os.makedirs(OUT, exist_ok=True)

def hours(): return (time.time() - T0) / 3600
def run(cmd, est_hours=0.0):
    if hours() + est_hours > TIME_BUDGET_HOURS:
        print(f"SKIPPED (time budget): {cmd[:90]}"); return False
    if SMOKE and any(k in cmd for k in ("s02_", "s03_", "s04_", "s05_")): cmd += " --smoke"
    print(f"\n[{hours():.2f} h] $ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True, cwd=REPO)
    if r.returncode != 0: print(f"!! step failed with code {r.returncode} - continuing")
    return r.returncode == 0

In [ ]:
# ---- locate the dataset, resume previous output, fetch figshare patient IDs --
DATA = None; PREV = []
for root, dirs, files in os.walk("/kaggle/input" if os.path.isdir("/kaggle/input") else WORK):
    low = {d.lower() for d in dirs}
    if DATA is None and {"training", "testing"} <= low:
        DATA = root
    if os.path.basename(root) == "results" and os.path.exists(os.path.join(root, "groups.csv")) and not root.startswith(WORK):
        PREV.append(root)
assert DATA, "Add the masoudnickparvar/brain-tumor-mri-dataset input first"
print("dataset        :", DATA)
for p in PREV:
    print("resuming from  :", p); shutil.copytree(p, OUT, dirs_exist_ok=True)

FIG = None
mats = glob.glob("/kaggle/input/**/*.mat", recursive=True) if os.path.isdir("/kaggle/input") else []
if len(mats) > 1000:
    FIG = os.path.dirname(mats[0])
else:
    try:
        import urllib.request
        dst_dir = os.path.join(WORK, "figshare")
        os.makedirs(dst_dir, exist_ok=True)
        meta = json.load(urllib.request.urlopen("https://api.figshare.com/v2/articles/1512427/files", timeout=60))
        for f in meta:
            if f["name"].endswith(".zip"):
                dst = os.path.join(dst_dir, f["name"])
                print("downloading", f["name"], f"{f['size']/1e6:.0f} MB")
                urllib.request.urlretrieve(f["download_url"], dst)
                with zipfile.ZipFile(dst) as z: z.extractall(os.path.join(dst_dir, "mat"))
                os.remove(dst)
        FIG = os.path.join(dst_dir, "mat")
    except Exception as e:
        print("!! figshare download failed:", e, "- patient-ID recovery skipped (turn Internet on)")
print("figshare .mat  :", FIG)

In [ ]:
# ---- experiments, in priority order -----------------------------------------
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
G = f"--data_root '{DATA}' --groups {OUT}/groups.csv --out {OUT} --cache_dir {WORK}/cache"
run(f"python scripts/s00_audit.py --data_root '{DATA}' --out {OUT}")
if not os.path.exists(f"{OUT}/dedup_summary.json"):
    run(f"python scripts/s01_dedup.py --data_root '{DATA}' --out {OUT}" + (f" --figshare_dir '{FIG}'" if FIG else ""), 0.5)
if not os.path.exists(f"{OUT}/dedup_098/dedup_summary.json"):
    run(f"python scripts/s01_dedup.py --data_root '{DATA}' --out {OUT}/dedup_098 --ncc_thr 0.98", 0.3)
run(f"python scripts/s02_cv.py {G} --models efficientnet_b0 --modes two_stage --seeds 0 1 2 --protocol grouped", 0.5)
run(f"python scripts/s03_test.py {G} --models efficientnet_b0 --modes two_stage --seeds 0 1 2", 0.4)
run(f"python scripts/s04_explain.py {G} --model efficientnet_b0 --mode two_stage --seed 0", 0.2)
run(f"python scripts/s02_cv.py {G} --models efficientnet_b0 --modes two_stage --seeds 0 --protocol naive", 0.5)
run(f"python scripts/s02_cv.py {G} --models resnet18 mobilenet_v3_large shufflenet_v2_x1_0 --modes two_stage --seeds 0 --protocol grouped", 1.0)
run(f"python scripts/s02_cv.py {G} --models efficientnet_b0 --modes frozen single_stage --seeds 0 --protocol grouped", 0.8)
run(f"python scripts/s03_test.py {G} --models resnet18 mobilenet_v3_large shufflenet_v2_x1_0 --modes two_stage --seeds 0", 0.3)
run(f"python scripts/s03_test.py {G} --models efficientnet_b0 --modes frozen single_stage --seeds 0", 0.3)
run(f"python scripts/s05_latency.py --out {OUT} --weights {OUT}/test/efficientnet_b0__two_stage__s0.pt")
print(f"experiments finished after {hours():.2f} h")

In [ ]:
# ---- summary values, figures and results.zip --------------------------------
subprocess.run(f"python scripts/s06_report.py --out {OUT} --data_root '{DATA}'", shell=True, cwd=REPO)
subprocess.run(f"python scripts/s07_source_analysis.py --out {OUT}", shell=True, cwd=REPO)
zpath = os.path.join(WORK, "results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            if f.endswith(".pt"): continue
            p = os.path.join(root, f); z.write(p, os.path.relpath(p, WORK))
print("DONE ->", zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)")
print(open(f"{OUT}/manuscript_values.md").read()[:3000])